In [1]:
import pandas as pd

In [2]:
matches = pd.read_csv('matches.csv', index_col=0)

In [3]:
matches.head()

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
1,2024-08-18,18:30,Serie A,Matchweek 1,Sun,Away,L,0.0,3.0,Hellas Verona,...,Match Report,NaN,14,4,17.8,0.0,0,0,2022,Napoli
2,2024-08-25,20:45,Serie A,Matchweek 2,Sun,Home,W,3.0,0.0,Bologna,...,Match Report,NaN,16,5,17.5,0.0,0,0,2022,Napoli
3,2024-08-31,20:45,Serie A,Matchweek 3,Sat,Home,W,2.0,1.0,Parma,...,Match Report,NaN,29,7,17.5,1.0,0,0,2022,Napoli
4,2024-09-15,18:00,Serie A,Matchweek 4,Sun,Away,W,4.0,0.0,Cagliari,...,Match Report,NaN,13,5,16.0,0.0,0,0,2022,Napoli
5,2024-09-21,18:00,Serie A,Matchweek 5,Sat,Away,D,0.0,0.0,Juventus,...,Match Report,NaN,8,1,23.4,0.0,0,0,2022,Napoli


In [4]:
del matches["comp"]
del matches["notes"]

In [7]:
matches["date"] = pd.to_datetime(matches["date"])
matches["target"] = (matches["result"] == "W").astype("int")

In [9]:
matches["venue_code"] = matches["venue"].astype("category").cat.codes
matches["opp_code"] = matches["opponent"].astype("category").cat.codes
matches["hour"] = matches["time"].str.replace(":.+", "", regex=True).astype("int")
matches["day_code"] = matches["date"].dt.dayofweek

In [11]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=50, min_samples_split=10, random_state=1)

In [12]:
train = matches[matches["date"] < '2024-10-03']
test = matches[matches["date"] > '2024-10-03']

In [15]:
predictors = ["venue_code", "opp_code", "hour", "day_code"]

In [16]:
rf.fit(train[predictors], train["target"])

RandomForestClassifier(min_samples_split=10, n_estimators=50, random_state=1)

In [17]:
preds = rf.predict(test[predictors])

In [18]:
from sklearn.metrics import accuracy_score
error = accuracy_score(test["target"], preds)
error

0.625

In [19]:
combined = pd.DataFrame(dict(actual=test["target"], predicted=preds))
pd.crosstab(index=combined["actual"], columns=combined["predicted"])

predicted,0,1
actual,,
0,2,1
1,2,3


In [20]:
from sklearn.metrics import precision_score
precision_score(test["target"], preds)

np.float64(0.75)

In [21]:
grouped_matches = matches.groupby("team")

In [22]:
def rolling_averages(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(3, closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset=new_cols)
    return group

In [23]:
cols = ["gf", "ga", "sh", "sot", "dist", "fk", "pk", "pkatt"]
new_cols = [f"{c}_rolling" for c in cols]

In [24]:
matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))

/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_10025/4052147919.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_rolling = matches.groupby("team").apply(lambda x: rolling_averages(x, cols, new_cols))


In [25]:
matches_rolling

date   time        round  day venue result   gf   ga  \
team                                                                          
Atalanta       4 2024-09-15  15:00  Matchweek 4  Sun  Home      W  3.0  2.0   
               6 2024-09-24  20:45  Matchweek 5  Tue  Home      L  2.0  3.0   
               7 2024-09-28  20:45  Matchweek 6  Sat  Away      D  1.0  1.0   
               9 2024-10-05  18:00  Matchweek 7  Sat  Home      W  5.0  1.0   
Internazionale 3 2024-09-15  20:45  Matchweek 4  Sun  Away      D  1.0  1.0   
               5 2024-09-22  20:45  Matchweek 5  Sun  Home      L  1.0  2.0   
               6 2024-09-28  15:00  Matchweek 6  Sat  Away      W  3.0  2.0   
               8 2024-10-05  20:45  Matchweek 7  Sat  Home      W  3.0  2.0   
Juventus       3 2024-09-14  18:00  Matchweek 4  Sat  Away      D  0.0  0.0   
               5 2024-09-21  18:00  Matchweek 5  Sat  Home      D  0.0  0.0   
               6 2024-09-28  18:00  Matchweek 6  Sat  Away      W  3.0  0.0   
               8 2024-10-06  12:30  Matchweek 7  Sun  Home      D  1.0  1.0   
Lazio          3 2024-09-16  20:45  Matchweek 4  Mon  Home      W  2.0  1.0   
               4 2024-09-22  12:30  Matchweek 5  Sun  Away      L  1.0  2.0   
               6 2024-09-29  12:30  Matchweek 6  Sun  Away      W  3.0  2.0   
               8 2024-10-06  15:00  Matchweek 7  Sun  Home      W  2.0  1.0   
Milan          3 2024-09-14  20:45  Matchweek 4  Sat  Home      W  4.0  0.0   
               5 2024-09-22  20:45  Matchweek 5  Sun  Away      W  2.0  1.0   
               6 2024-09-27  20:45  Matchweek 6  Fri  Home      W  3.0  0.0   
               8 2024-10-06  20:45  Matchweek 7  Sun  Away      L  1.0  2.0   
Napoli         4 2024-09-15  18:00  Matchweek 4  Sun  Away      W  4.0  0.0   
               5 2024-09-21  18:00  Matchweek 5  Sat  Away      D  0.0  0.0   
               7 2024-09-29  20:45  Matchweek 6  Sun  Home      W  2.0  0.0   
               8 2024-10-04  18:30  Matchweek 7  Fri  Home      W  3.0  1.0   
Torino         4 2024-09-15  15:00  Matchweek 4  Sun  Home      D  0.0  0.0   
               5 2024-09-20  20:45  Matchweek 5  Fri  Away      W  3.0  2.0   
               7 2024-09-29  12:30  Matchweek 6  Sun  Home      L  2.0  3.0   
               8 2024-10-05  20:45  Matchweek 7  Sat  Away      L  2.0  3.0   
Udinese        4 2024-09-16  18:30  Matchweek 4  Mon  Away      W  3.0  2.0   
               5 2024-09-22  18:00  Matchweek 5  Sun  Away      L  0.0  3.0   
               7 2024-09-28  15:00  Matchweek 6  Sat  Home      L  2.0  3.0   
               8 2024-10-05  15:00  Matchweek 7  Sat  Home      W  1.0  0.0   

                       opponent   xg  ...  hour  day_code  gf_rolling  \
team                                  ...                               
Atalanta       4     Fiorentina  1.7  ...    15         6    1.666667   
               6           Como  1.6  ...    20         1    1.333333   
               7        Bologna  2.0  ...    20         5    1.666667   
               9          Genoa  3.6  ...    18         5    2.000000   
Internazionale 3          Monza  1.7  ...    20         6    2.666667   
               5          Milan  0.7  ...    20         6    2.333333   
               6        Udinese  2.2  ...    15         5    2.000000   
               8         Torino  2.4  ...    20         5    1.666667   
Juventus       3         Empoli  0.9  ...    18         5    2.000000   
               5         Napoli  0.3  ...    18         5    1.000000   
               6          Genoa  2.6  ...    18         5    0.000000   
               8       Cagliari  2.4  ...    12         6    1.000000   
Lazio          3  Hellas Verona  1.2  ...    20         0    2.000000   
               4     Fiorentina  1.0  ...    12         6    1.666667   
               6         Torino  2.0  ...    12         6    1.666667   
               8         Empoli  2.1  ...    15         6    2.000000   
Milan          3      

In [26]:
matches_rolling = matches_rolling.droplevel('team')
matches_rolling.index = range(matches_rolling.shape[0])

In [29]:
def make_predictions(data, predictors):
    train = data[data["date"] < '2024-10-03']
    test = data[data["date"] > '2024-10-03']
    rf.fit(train[predictors], train["target"])
    preds = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual=test["target"], predicted=preds), index=test.index)
    error = precision_score(test["target"], preds)
    return combined, error

In [30]:
combined, error = make_predictions(matches_rolling, predictors + new_cols)
error

np.float64(0.6)

In [32]:
combined = combined.merge(matches_rolling[["date", "team", "opponent", "result"]], left_index=True, right_index=True)
combined

,actual,predicted,date_x,team_x,opponent_x,result_x,date_y,team_y,opponent_y,result_y
3,1,1,2024-10-05,Atalanta,Genoa,W,2024-10-05,Atalanta,Genoa,W
7,1,1,2024-10-05,Internazionale,Torino,W,2024-10-05,Internazionale,Torino,W
11,0,0,2024-10-06,Juventus,Cagliari,D,2024-10-06,Juventus,Cagliari,D
15,1,1,2024-10-06,Lazio,Empoli,W,2024-10-06,Lazio,Empoli,W
19,0,1,2024-10-06,Milan,Fiorentina,L,2024-10-06,Milan,Fiorentina,L
23,1,0,2024-10-04,Napoli,Como,W,2024-10-04,Napoli,Como,W
27,0,1,2024-10-05,Torino,Inter,L,2024-10-05,Torino,Inter,L
31,1,0,2024-10-05,Udinese,Lecce,W,2024-10-05,Udinese,Lecce,W
